In [8]:
import pandas as pd
import numpy as np

caucasusUncertainty = pd.read_csv('../Data/Processed/weeklyUncertaintyCaucasusCA.csv')
europeUncertainty = pd.read_csv('../Data/Processed/weeklyUncertaintyEurope.csv')
middleEastUncertainty = pd.read_csv('../Data/Processed/weeklyUncertaintyMiddleEast.csv')
northAfricaUncertainty = pd.read_csv('../Data/Processed/weeklyUncertaintyNorthAfrica.csv')

allUncertainty = pd.concat([caucasusUncertainty, europeUncertainty, middleEastUncertainty, northAfricaUncertainty])

print(allUncertainty.shape)
print(allUncertainty['region'].value_counts())

(371059, 8)
region
Europe                       320620
Middle East                   30498
Caucasus and Central Asia     14076
Northern Africa                5865
Name: count, dtype: int64


In [9]:
caucasusUncertainty['entropyReliable'] = True
middleEastUncertainty['entropyReliable'] = True
northAfricaUncertainty['entropyReliable'] = True

caucasusUncertainty.to_csv('../Data/Processed/weeklyUncertaintyCaucasusCA.csv', index=False)
middleEastUncertainty.to_csv('../Data/Processed/weeklyUncertaintyMiddleEast.csv', index=False)
northAfricaUncertainty.to_csv('../Data/Processed/weeklyUncertaintyNorthAfrica.csv', index=False)

print(caucasusUncertainty['entropyReliable'].unique())

[ True]


In [10]:
allUncertainty['distanceFromCertainty'] = 1 - np.abs(2 * allUncertainty['flipProbability'] - 1)

weeklyComposite = allUncertainty.groupby(['week', 'region']).agg(
    meanUncertainty = ('distanceFromCertainty', 'mean'),
    hdiOverlapRate = ('hdiOverlap', 'mean'), 
    entropy = ('entropy', 'first'), 
    entropyReliable = ('entropyReliable', 'first')
).reset_index()

print(weeklyComposite.shape) 
weeklyComposite.head()

(1564, 6)


,week,region,meanUncertainty,hdiOverlapRate,entropy,entropyReliable
0,2018-01-01/2018-01-07,Caucasus and Central Asia,0.151944,0.500000,4.175930,True
1,2018-01-01/2018-01-07,Europe,0.741524,0.956098,5.991465,False
2,2018-01-01/2018-01-07,Middle East,0.088013,0.230769,4.774322,True
3,2018-01-01/2018-01-07,Northern Africa,0.230667,0.600000,2.598303,True
4,2018-01-08/2018-01-14,Caucasus and Central Asia,0.090972,0.444444,3.396296,True


In [11]:
weeklyComposite.to_csv('../Data/Processed/weeklyCompositeUncertainty.csv', index=False)

In [13]:
alignedData = pd.read_csv('../Data/Processed/alignedData.csv')

marketEventDay = alignedData[alignedData['dayOffset'] == 0].copy()

if marketEventDay['tradingDay'].dtype != 'datetime64[ns]':
    marketEventDay['tradingDay'] = pd.to_datetime(marketEventDay['tradingDay'])

marketEventDay['week'] = marketEventDay['tradingDay'].dt.to_period('W')

print(marketEventDay.shape)
print(marketEventDay.columns.tolist())

(1878, 12)
['tradingDay', 'dayOffset', 'events_CaucasusCA', 'events_Europe', 'events_MiddleEast', 'events_NorthAfrica', 'Return_EEM', 'Return_GLD', 'Return_ITA', 'Return_USO', 'Return_VIX', 'week']


In [14]:
returnCols = ['Return_EEM', 'Return_GLD', 'Return_ITA', 'Return_USO', 'Return_VIX']

weeklyMarket = marketEventDay.groupby('week')[returnCols].sum().reset_index()

print(weeklyMarket.shape)
weeklyMarket.head()

(391, 6)


,week,Return_EEM,Return_GLD,Return_ITA,Return_USO,Return_VIX
0,2018-01-01/2018-01-07,0.023155,0.001454,0.017479,0.019950,-0.055809
1,2018-01-08/2018-01-14,0.007791,0.013014,0.036263,0.044748,0.100018
2,2018-01-15/2018-01-21,0.018738,-0.004216,0.000336,-0.011679,0.117366
3,2018-01-22/2018-01-28,0.032154,0.013078,0.026043,0.040539,-0.015204
4,2018-01-29/2018-02-04,-0.059267,-0.013065,-0.018393,-0.014710,0.513131


In [16]:
weeklyComposite['week'] = pd.PeriodIndex(weeklyComposite['week'], freq='W-SUN')

mergedData = weeklyComposite.merge(weeklyMarket, on='week', how='inner')

print(mergedData.shape)
mergedData.head()

(1564, 11)


,week,region,meanUncertainty,hdiOverlapRate,entropy,entropyReliable,Return_EEM,Return_GLD,Return_ITA,Return_USO,Return_VIX
0,2018-01-01/2018-01-07,Caucasus and Central Asia,0.151944,0.500000,4.175930,True,0.023155,0.001454,0.017479,0.019950,-0.055809
1,2018-01-01/2018-01-07,Europe,0.741524,0.956098,5.991465,False,0.023155,0.001454,0.017479,0.019950,-0.055809
2,2018-01-01/2018-01-07,Middle East,0.088013,0.230769,4.774322,True,0.023155,0.001454,0.017479,0.019950,-0.055809
3,2018-01-01/2018-01-07,Northern Africa,0.230667,0.600000,2.598303,True,0.023155,0.001454,0.017479,0.019950,-0.055809
4,2018-01-08/2018-01-14,Caucasus and Central Asia,0.090972,0.444444,3.396296,True,0.007791,0.013014,0.036263,0.044748,0.100018


In [17]:
mergedData.to_csv('../Data/Processed/mergedUncertaintyMarket.csv', index=False)

In [19]:
for region in mergedData['region'].unique(): 
    regionData = mergedData[mergedData['region'] == region]
    correlation = regionData['meanUncertainty'].corr(regionData['Return_VIX'])
    print(f'{region}: correlation with same week VIX return = {correlation:4f}')

Caucasus and Central Asia: correlation with same week VIX return = -0.035061
Europe: correlation with same week VIX return = 0.003830
Middle East: correlation with same week VIX return = -0.126229
Northern Africa: correlation with same week VIX return = -0.010635


In [22]:
for region in mergedData['region'].unique(): 
    regionData = mergedData[mergedData['region'] == region]

    print(f'--{region}--')
    for lag in range(0, 4): 
        laggedVIX = regionData['Return_VIX'].shift(-lag)
        correlation = regionData['meanUncertainty'].corr(laggedVIX)
        print(f'Lag {lag}: Correlation = {correlation:.4f}')

--Caucasus and Central Asia--
Lag 0: Correlation = -0.0351
Lag 1: Correlation = -0.0352
Lag 2: Correlation = -0.0090
Lag 3: Correlation = -0.0402
--Europe--
Lag 0: Correlation = 0.0038
Lag 1: Correlation = 0.0152
Lag 2: Correlation = 0.0117
Lag 3: Correlation = 0.0075
--Middle East--
Lag 0: Correlation = -0.1262
Lag 1: Correlation = 0.0170
Lag 2: Correlation = -0.0311
Lag 3: Correlation = -0.0627
--Northern Africa--
Lag 0: Correlation = -0.0106
Lag 1: Correlation = 0.0070
Lag 2: Correlation = 0.0547
Lag 3: Correlation = 0.0689


In [23]:
metrics = ['meanUncertainty', 'hdiOverlapRate', 'entropy']
assets = ['Return_EEM', 'Return_GLD', 'Return_ITA', 'Return_USO', 'Return_VIX']
lags = range(0, 4)

sweepResults = []

for region in mergedData['region'].unique():
    regionData = mergedData[mergedData['region'] == region].sort_values('week').reset_index(drop=True)

    for metric in metrics:
        if metric == 'entropy' and not regionData['entropyReliable'].iloc[0]:
            continue

        for asset in assets:
            for lag in lags:
                laggedAsset = regionData[asset].shift(-lag)
                correlation = regionData[metric].corr(laggedAsset)

                sweepResults.append({
                    'region': region,
                    'metric': metric,
                    'asset': asset,
                    'lag': lag,
                    'correlation': correlation
                })

sweepResultsDF = pd.DataFrame(sweepResults)
print(sweepResultsDF.shape)

(220, 5)


In [24]:
sweepResultsDF['absCorrelation'] = sweepResultsDF['correlation'].abs()

topResults = sweepResultsDF.sort_values('absCorrelation', ascending=False).head(20)

print(topResults[['region', 'metric', 'asset', 'lag', 'correlation']].to_string(index=False))

         region          metric      asset  lag  correlation
    Middle East  hdiOverlapRate Return_VIX    0    -0.145826
Northern Africa  hdiOverlapRate Return_GLD    2    -0.129603
    Middle East meanUncertainty Return_VIX    0    -0.126229
    Middle East         entropy Return_VIX    0    -0.123307
Northern Africa         entropy Return_USO    3    -0.123072
Northern Africa         entropy Return_EEM    3    -0.122162
Northern Africa meanUncertainty Return_USO    3    -0.121883
Northern Africa         entropy Return_GLD    2    -0.111810
    Middle East         entropy Return_EEM    0     0.110447
    Middle East meanUncertainty Return_EEM    0     0.100824
Northern Africa meanUncertainty Return_EEM    3    -0.098128
Northern Africa  hdiOverlapRate Return_EEM    3    -0.096876
    Middle East meanUncertainty Return_EEM    3     0.090189
Northern Africa  hdiOverlapRate Return_GLD    3    -0.090123
Northern Africa         entropy Return_VIX    3     0.089008
    Middle East  hdiOver

In [25]:
middleEastData = mergedData[mergedData['region'] == 'Middle East'].sort_values('week').reset_index(drop=True)

earlyPeriod = middleEastData[middleEastData['week'] < pd.Period('2022-01-01', freq='W-SUN')]
latePeriod = middleEastData[middleEastData['week'] >= pd.Period('2022-01-01', freq='W-SUN')]

earlyCorr = earlyPeriod['hdiOverlapRate'].corr(earlyPeriod['Return_VIX'])
lateCorr = latePeriod['hdiOverlapRate'].corr(latePeriod['Return_VIX'])

print(f"Full period (2018-2025): -0.1458")
print(f"Early period (2018-2021): {earlyCorr:.4f}, n={len(earlyPeriod)}")
print(f"Late period (2022-2025): {lateCorr:.4f}, n={len(latePeriod)}")

Full period (2018-2025): -0.1458
Early period (2018-2021): -0.0627, n=208
Late period (2022-2025): -0.2375, n=183
